# Imports

In [ ]:
#import packages
import numpy as np
import json 
import scanpy as sc
from collections import OrderedDict
import scipy 
import pandas as pd
import matplotlib.pyplot as plt
import pickle

#spectra imports 
import Spectra 
from Spectra import Spectra_util as spc_tl
from Spectra import K_est as kst
from Spectra import default_gene_sets

In [ ]:
pwd

In [ ]:
adata = sc.read_h5ad('/Users/mkaur/github/2_tff1/1_tecs/1_pyzone/3_h5ad/8_adata_final.h5ad')
adata

In [ ]:
sc.set_figure_params(dpi=100, dpi_save=300, color_map='Spectral_r', vector_friendly=True, transparent=True)
sc.pl.umap(
    adata, 
    color=['cell_type_subset', 'sample', 'cell_type'],  
    ncols=1,
    size =5, 
    outline_width=[0.6, 0.05],
    frameon=False,
    cmap='Spectral_r',
    use_raw=False, 
    #vmax = 'p99', 
    add_outline=True,
    #legend_loc = 'on data', 
    title='',
    #save='_annotation.pdf'
)

# Set up Spectra Dict

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='cell_type_subset', method='wilcoxon', use_raw=False)
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(100) #'scores', 'logfoldchanges', 'pvals_adj'

df.head()

In [ ]:
N = 35  # how many markers per cell type

gene_set_dictionary = {}
for ct in groups:
    # Grab the top N names, ensure strings, drop NaNs/empties, keep unique order
    names = pd.Series(result['names'][ct]).astype(str)
    names = names.replace({'nan': np.nan}).dropna()
    # Keep order-unique
    seen = set()
    topN = [g for g in names.tolist() if (g not in seen and not seen.add(g))][:N]
    
    # One gene set per cell type
    gene_set_dictionary[ct] = {f'{ct}': topN}

In [ ]:
gene_set_dictionary

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='cell_type', method='wilcoxon', use_raw=False)
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names']}).head(100) #'scores', 'logfoldchanges', 'pvals_adj'

df.head()

In [ ]:
#df['vSMC/PC_n'].head(35).to_list()

In [ ]:
gene_set_dictionary['global'] = {
    'epithelial': ['Epcam',
 'Dsp',
 'Cdh1',
 'Rbm47',
 'Spint2',
 'Alcam',
 'Krt8',
 'Krt18',
 'H2-Eb1',
 'Ank3',
 'Gm15987',
 'Esrp1',
 'Spint1',
 'Patj',
 'Slc9a9',
 'H2-Ab1',
 'Map7',
 'Cd74',
 'Dsg2',
 'Cdcp1',
 'Sgpl1',
 'Tiam1',
 'H2-Aa',
 'Sorl1',
 'Eya4',
 'Pstpip2',
 'Parm1',
 'Ctnnd2',
 'Eya2',
 'Pclo',
 'Sfn',
 'Wwc1',
 'Mreg',
 'Snx29',
 'Tmem131l'],
    'fibroblast': ['Col6a3',
 'Lama2',
 'Col5a2',
 'Ank2',
 'Lhfp',
 'Col6a2',
 'Fbn1',
 'Col3a1',
 'Lum',
 'Col6a1',
 'Col1a2',
 'Ddr2',
 'Abi3bp',
 'Col1a1',
 'Bicc1',
 'Prrx1',
 'Zeb2',
 'Dlc1',
 'Bgn',
 'Dcn',
 'Apod',
 'Colec12',
 'Gsn',
 'Col5a1',
 'Col5a3',
 'Cd302',
 'Gpm6b',
 'Htra3',
 'Il1r1',
 'Pdgfra',
 'Mxra8',
 'Serpine2',
 'Enpp2',
 'Serping1',
 'Cygb'], 
    'endothelial': ['Ptprm',
 'Sptbn1',
 'Pecam1',
 'Cd36',
 'Egfl7',
 'Cdh5',
 'Adgrf5',
 'Flt1',
 'Arhgap31',
 'Adgrl4',
 'Eng',
 'Cav1',
 'Fli1',
 'Ptprb',
 'Nrp1',
 'Ly6c1',
 'Col4a1',
 'Prkch',
 'Shank3',
 'Kdr',
 'Prex2',
 'Epas1',
 'Apbb2',
 'Cavin2',
 'Esam',
 'Cyyr1',
 'Itga1',
 'S1pr1',
 'Col4a2',
 'Tcf4',
 'Emcn',
 'Ushbp1',
 'Ldb2',
 'Arhgap29',
 'Gng11'], 
    'mesothelial': ['Aebp1',
 'Csrp2',
 'Igfbp6',
 'Upk3b',
 'Dcn',
 'Rarres2',
 'C3',
 'Serping1',
 'Gpm6a',
 'Timp2',
 'Mgp',
 'Igfbp4',
 'C4b',
 'Aldh1a2',
 'Ptgis',
 'Abi1',
 'Lgals1',
 'C2',
 'Nkain4',
 'Efemp1',
 'Fbln1',
 'Pals2',
 'Wt1',
 'Rspo1',
 'Sox6',
 'Rnase4',
 'Cldn15',
 'Col1a2',
 'Upk1b',
 'Gata6',
 'Gm56730',
 'Fndc1',
 'Lrrn4',
 'Laptm4a',
 'Ccdc80'],
    'vSMC/PC':['Prkg1',
 'Zeb2',
 'Dlc1',
 'Gucy1b1',
 'Nr2f2',
 'Ednra',
 'Pde3a',
 'Plac9',
 'Sox5',
 'Sparcl1',
 'Gucy1a1',
 'Cacna1c',
 'Gjc1',
 'Pdgfrb',
 'Ndufa4l2',
 'Ebf1',
 'Lhfp',
 'Bgn',
 'Rhoj',
 'Prrx1',
 'Pde1a',
 'Mylk',
 'Gm13889',
 'Plcl1',
 'Serping1',
 'Notch3',
 'Arhgap10',
 'Ppp1r12a',
 'Rgs5',
 'Rarres2',
 'Cald1',
 'Myl9',
 'Sparc',
 'Pla2r1',
 'Lin7a']
                        }

In [ ]:
#filter gene set annotation dict for genes contained in adata
annotations = Spectra.Spectra_util.check_gene_set_dictionary(adata,
                                                             gene_set_dictionary,
                                                             obs_key='cell_type_subset',
                                                             global_key='global')

In [ ]:
#adata.X = adata.X.toarray()

In [ ]:
model = Spectra.est_spectra(adata=adata, 
                            gene_set_dictionary=gene_set_dictionary, 
                            use_highly_variable=False,
                            cell_type_key="cell_type_subset", 
                            use_weights=True,
                            lam=0.1, 
                            rho=0.001, 
                            use_cell_types=True,
                            n_top_vals=35,
                            label_factors=True, 
                            overlap_threshold=0.2,
                            clean_gs = True, 
                            min_gs_num = 3,
                            num_epochs=100 # For a more serious analysis, please make this 10000
                            )

In [ ]:
model.return_eta_diag()

In [ ]:
import pickle
with open('/coh_labs/mvandenbrink/users/pkaur/6_tff1/0_scRNA/1_spectra/2_outputs/0_files/tec_all_spectra_model.pickle', 'wb') as f:
    pickle.dump(model, f, pickle.HIGHEST_PROTOCOL)

In [ ]:
adata

In [ ]:
adata.write_h5ad('/coh_labs/mvandenbrink/users/pkaur/6_tff1/0_scRNA/1_spectra/2_outputs/adata_all_spectra.h5ad')